# Generic optimizer research notebook

This notebook targets any optimizer runner module that exposes an `OPTIMIZER` object. Today's public examples use `optimizer_type=\"parameter_search\"`. The notebook runs a compact sweep, then replays one selected window with the winning parameters and emits HTML into `output/`.

Launch it from `make backtest` or `uv run python main.py` so the repo menu treats it like any other flat runner entrypoint.


In [ ]:
import importlib
from dataclasses import replace
from pprint import pprint

from prediction_market_extensions.backtesting.optimizers import run_parameter_search

OPTIMIZER_MODULE = "backtests.polymarket_quote_tick_pmxt_ema_optimizer"
MAX_TRIALS = 2
HOLDOUT_TOP_K = 1

optimizer_module = importlib.import_module(OPTIMIZER_MODULE)
optimizer_config = getattr(
    optimizer_module,
    "OPTIMIZER",
    getattr(
        optimizer_module,
        "PARAMETER_SEARCH",
        getattr(optimizer_module, "OPTIMIZATION"),
    ),
)
parameter_search = replace(
    optimizer_config,
    name=f"{optimizer_config.name}_research",
    max_trials=min(MAX_TRIALS, optimizer_config.max_trials),
    holdout_top_k=min(HOLDOUT_TOP_K, optimizer_config.holdout_top_k),
)

summary = run_parameter_search(parameter_search)
best_params = dict(summary.selected_params)
print("Optimizer module:", OPTIMIZER_MODULE)
print("Selected params:")
pprint(best_params)
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

In [ ]:
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)

selected_window = (
    parameter_search.holdout_windows[0]
    if parameter_search.holdout_windows
    else parameter_search.train_windows[-1]
)
backtest = build_parameter_search_window_backtest(
    config=parameter_search,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{parameter_search.name}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path="output",
    return_summary_series=False,
)

results = backtest.run()
print(f"Replayed {len(results)} market(s) for {selected_window.name}.")
print("HTML output root:", backtest.chart_output_path)

<!-- prediction-market-backtesting:auto-embedded-html -->
Generated HTML artifacts will be embedded here after execution.
